# eph_06 — Behavioral comparison: RT encoding vs Sue model outputs

Compares per-unit RT encoding T-statistics with AIND behavioral model ("Sue")
encodings, in **both** the response (0–0.2 s post-cue) and baseline (−1–0 s)
windows. Three task variables are pinned to match the poster figures:

| Poster label | Sue column |
|---|---|
| Value Prediction | `sue::Qchosen_l_mc` |
| Reward Outcome | `sue::outcome_l_mc` |
| Respond vs Ignore | `sue::response_hit_all` (response) / `sue::baseline_hit_all` (baseline) |

**Pipeline:**
1. Fit RT encoding in the response and baseline windows (our own `fit_encoding`),
   register both in `PerUnitStatsRegistry`
2. Bulk-register all Sue T-columns via `register_sue`
3. Screen RT T-stats against all Sue columns (Spearman ρ of T-stat vectors)
4. Pinned `registry_compare_plot` scatters for the three task variables × two windows
5. Combined 2×3 `rt_vs_task_6panel` summary
6. T-scatter + polar histogram for key pairs (outcome, Q-chosen, baseline)

## 1. Setup

In [ ]:
%matplotlib inline
import contextlib, io
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from plotstyle import apply_style, PALETTE, style_ax, save_fig
apply_style()

In [ ]:
from pathlib import Path

if Path("/root/capsule").exists():
    ENV       = "codeocean"
    SCRATCH   = Path("/root/capsule/scratch")
    DATA_ROOT = Path("/root/capsule/data")
    FOR_LOCAL = SCRATCH / "for_local"
else:
    ENV       = "local"
    FOR_LOCAL = Path("/Users/mib/Documents/Code/kinematics_analysis/data/for_local")
    DATA_ROOT = FOR_LOCAL
    SCRATCH   = FOR_LOCAL.parent

FIG_DIR  = SCRATCH / "figures" / "eph_06_behavioral_comparison"
SAVE_FIG = False
print(f"ENV={ENV}  FOR_LOCAL={FOR_LOCAL}")

## 2. Data loading

In [ ]:
from data_loading import (
    load_session_quality_filter, filter_ephys_units, load_units_with_spike_times,
)
import pickle

if ENV == "codeocean":
    base_dirs = [SCRATCH / "session_analysis_mlk"]
    filtered_session_paths = load_session_quality_filter(base_dirs)
    with open(SCRATCH / "combined_unit_tbl.pkl", "rb") as f:
        combined_ephys_data = pickle.load(f)
    filtered_ephys = filter_ephys_units(combined_ephys_data, filtered_session_paths)
    ROOT_SCRATCH = str(DATA_ROOT / "LC-NE_scratch_data_1")
    units_with_spikes = load_units_with_spike_times(filtered_ephys, ROOT_SCRATCH)
else:
    filtered_ephys    = pd.read_pickle(FOR_LOCAL / "filtered_ephys.pkl")
    units_with_spikes = None
    base_dirs         = [FOR_LOCAL]
    print(f"Local dev: filtered_ephys {filtered_ephys.shape}")

In [ ]:
from ephys_utils import AnalysisConfig, build_all_counts_df

cfg = AnalysisConfig(
    align_key="goCue",
    count_window_s=(0.0, 0.2),
    baseline_window_s=(-1.0, 0.0),
    min_trials_per_group=20,
)
if ENV == "codeocean":
    all_counts_df = build_all_counts_df(units_with_spikes, cfg, base_dirs)
else:
    all_counts_df = pd.read_parquet(FOR_LOCAL / "all_counts_df.parquet")
print("all_counts_df:", all_counts_df.shape)

## 3. Imports

In [ ]:
from encoding_methods import AnalysisSpec, fit_encoding
from per_unit_stats_registry import PerUnitStatsRegistry
from encoding_plots import registry_compare_plot, tstat_hist
from aind_dynamic_foraging_behavior_video_analysis.ephys.tongue_ephys import get_session_prefix
import contextlib, io

## 4. Load Sue behavioral model table

In [ ]:
# Load Sue behavioral model encoding table
if ENV == "codeocean":
    sue_path = SCRATCH / "features_combined_beh_all.pkl"
else:
    sue_path = FOR_LOCAL / "features_combined_beh_all.pkl"

sue_features = pd.read_pickle(sue_path)
print("sue_features shape:", sue_features.shape)
print("T_ columns:", [c for c in sue_features.columns if str(c).startswith("T_")][:8], "...")

## 5. Fit RT encoding and register

In [ ]:
# Fit RT encoding in BOTH windows (same RT spec as eph_01, two response columns).
# Baseline-window counts already live in all_counts_df as `baseline_spike_count`.
RT_RESPONSE_SPEC = AnalysisSpec(
    name="ols_rt_response",
    predictor_col="reaction_time_firstmove",
    response_col="spike_count",
    method="ols",
    trial_query="reaction_time_firstmove > 0.05 and reaction_time_firstmove < 1.5",
    log_x=True, zscore_x=True,
    notes="ols: response-window spike_count ~ log(RT)",
)
RT_BASELINE_SPEC = AnalysisSpec(
    name="ols_rt_baseline",
    predictor_col="reaction_time_firstmove",
    response_col="baseline_spike_count",
    method="ols",
    trial_query="reaction_time_firstmove > 0.05 and reaction_time_firstmove < 1.5",
    log_x=True, zscore_x=True,
    notes="ols: baseline-window spike_count ~ log(RT)",
)
with contextlib.redirect_stdout(io.StringIO()):
    rt_response_result = fit_encoding(all_counts_df, RT_RESPONSE_SPEC)
    rt_baseline_result = fit_encoding(all_counts_df, RT_BASELINE_SPEC)

rt_result = rt_response_result  # alias used by the sue_plus merge below
print(f"RT response n_sig: {rt_response_result.n_sig()}")
print(f"RT baseline n_sig: {rt_baseline_result.n_sig()}")

In [ ]:
reg = PerUnitStatsRegistry(get_session_prefix=get_session_prefix, alpha=0.05)
reg.register(rt_response_result)
reg.register(rt_baseline_result)

# Bulk-register all Sue T_ columns as "sue::{suffix}" entries
n_sue = reg.register_sue(
    sue_features,
    t_prefix="T_", p_prefix="p_", coef_prefix="coef_",
    registry_prefix="sue",
)
print(f"Registered {n_sue} Sue entries")
print(reg)

## 6. Screen: which Sue T-columns correlate most with RT encoding?

In [ ]:
# Screen response-window RT T-stats against all Sue T_ columns via Spearman T-T correlation.
# Exploratory context for the pinned comparisons below.
screen = reg.screen(
    "ols_rt_response",
    source="sue",
    min_n=10,
    rank_by="abs_rho",
    top_n=20,
)
print("Top 20 Sue variables correlated with response-window RT encoding T-stats:")
print(screen[["entry","n","rho","p","abs_rho","fisher_OR","fisher_p"]].to_string(index=False))

## 7. Pinned RT-vs-task comparison scatters

Three task variables (Value Prediction, Reward Outcome, Respond vs Ignore),
each compared against RT encoding in the response and baseline windows — the six
`rt_{response,baseline}_vs_*` poster figures.

In [ ]:
# Pinned task variables (matching the poster). RT encoding is compared against
# three Sue model encodings in BOTH the response and baseline windows.
NICE = {
    "ols_rt_response":       "RT (response)",
    "ols_rt_baseline":       "RT (baseline)",
    "sue::Qchosen_l_mc":     "Value Prediction",
    "sue::outcome_l_mc":     "Reward Outcome",
    "sue::response_hit_all": "Respond vs Ignore",
    "sue::baseline_hit_all": "Respond vs Ignore",
}

# (rt_entry, sue_entry, save-name) — note Respond/Ignore uses the window-matched
# Sue column (response_hit_all for response, baseline_hit_all for baseline).
COMPARISONS = [
    ("ols_rt_response", "sue::Qchosen_l_mc",     "rt_response_vs_value_prediction"),
    ("ols_rt_response", "sue::outcome_l_mc",     "rt_response_vs_reward_outcome"),
    ("ols_rt_response", "sue::response_hit_all", "rt_response_vs_respond_ignore"),
    ("ols_rt_baseline", "sue::Qchosen_l_mc",     "rt_baseline_vs_value_prediction"),
    ("ols_rt_baseline", "sue::outcome_l_mc",     "rt_baseline_vs_reward_outcome"),
    ("ols_rt_baseline", "sue::baseline_hit_all", "rt_baseline_vs_respond_ignore"),
]

for rt_entry, sue_entry, fname in COMPARISONS:
    if rt_entry not in reg.names or sue_entry not in reg.names:
        print(f"  skip {fname}: missing registry entry ({rt_entry} / {sue_entry})")
        continue
    merged = reg.compare(rt_entry, sue_entry)
    # Relabel sig_category from registry keys to nice names so the legend matches the axes.
    merged["sig_category"] = (
        merged["sig_category"]
        .str.replace(rt_entry, NICE[rt_entry], regex=False)
        .str.replace(sue_entry, NICE[sue_entry], regex=False)
    )
    fig, *_ = registry_compare_plot(
        merged, NICE[rt_entry], NICE[sue_entry],
        title=f"{NICE[rt_entry]}  vs  {NICE[sue_entry]}",
    )
    save_fig(fig, fname, fig_dir=FIG_DIR, save=SAVE_FIG)
    plt.show()

## 7b. Combined 6-panel summary (`rt_vs_task_6panel`)

Top row = response window, bottom row = baseline window; columns = Value
Prediction, Reward Outcome, Respond vs Ignore. Points colored by which encoding
is FDR-significant (RT only / task only / both / neither).

In [ ]:
# Combined 2x3 summary: RT encoding (response top row / baseline bottom row)
# vs the three task variables. Reproduces the poster's rt_vs_task_6panel.
from scipy.stats import spearmanr, fisher_exact

COLOR_X    = PALETTE["pos"]      # RT-only significant
COLOR_Y    = PALETTE["neg"]      # task-variable-only significant
COLOR_BOTH = PALETTE["accent"]   # both significant
COLOR_NONE = "#d0d0d0"           # neither

PANEL_COMPARISONS = [
    ("ols_rt_response", "sue::Qchosen_l_mc"),
    ("ols_rt_response", "sue::outcome_l_mc"),
    ("ols_rt_response", "sue::response_hit_all"),
    ("ols_rt_baseline", "sue::Qchosen_l_mc"),
    ("ols_rt_baseline", "sue::outcome_l_mc"),
    ("ols_rt_baseline", "sue::baseline_hit_all"),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for idx, (rt_entry, sue_entry) in enumerate(PANEL_COMPARISONS):
    row, col = divmod(idx, 3)
    ax = axes[row, col]
    rt_label, sue_label = NICE[rt_entry], NICE[sue_entry]

    if rt_entry not in reg.names or sue_entry not in reg.names:
        ax.set_visible(False)
        continue

    merged = reg.compare(rt_entry, sue_entry)
    ta = merged["t_a"].to_numpy(dtype=float)
    tb = merged["t_b"].to_numpy(dtype=float)
    sa = merged["sig_fdr_a"].to_numpy().astype(bool)
    sb = merged["sig_fdr_b"].to_numpy().astype(bool)

    cats = np.full(len(merged), "neither", dtype=object)
    cats[sa & sb]  = "both"
    cats[sa & ~sb] = rt_label
    cats[~sa & sb] = sue_label

    nn = len(merged)
    rho, rho_p = spearmanr(ta, tb) if nn >= 3 else (np.nan, np.nan)
    contingency = np.array([
        [int((~sa & ~sb).sum()), int((~sa &  sb).sum())],
        [int(( sa & ~sb).sum()), int(( sa &  sb).sum())],
    ])
    odds_ratio, fisher_p_val = fisher_exact(contingency)

    cat_colors = {"neither": COLOR_NONE, rt_label: COLOR_X,
                  sue_label: COLOR_Y, "both": COLOR_BOTH}
    draw_order = ["neither", rt_label, sue_label, "both"]
    for cat in draw_order:
        mask = cats == cat
        if not mask.any():
            continue
        # col 0 shows all four categories; other cols show only the task-variable count
        if col == 0:
            label = f"{cat} ({int(mask.sum())})"
        elif cat == sue_label:
            label = f"{sue_label} ({int(mask.sum())})"
        else:
            label = None
        ax.scatter(ta[mask], tb[mask], c=cat_colors[cat], label=label,
                   s=25, alpha=0.7, edgecolors="none")

    if nn >= 2:
        z = np.polyfit(ta, tb, 1)
        x_line = np.linspace(ta.min(), ta.max(), 100)
        ax.plot(x_line, np.polyval(z, x_line), color="black", lw=1.2, alpha=0.6)
    ax.axhline(0, ls=":", color="grey", lw=0.5)
    ax.axvline(0, ls=":", color="grey", lw=0.5)
    ax.set_xlabel(f"t  ({rt_label})", fontsize=11)
    ax.set_ylabel(f"t  ({sue_label})", fontsize=11)
    ax.text(0.03, 0.97,
            f"n={nn}, ρ={rho:.3f}, p={rho_p:.2g}\n"
            f"Fisher OR={odds_ratio:.2f}, p={fisher_p_val:.2g}",
            transform=ax.transAxes, fontsize=8, va="top", ha="left")
    if ax.get_legend_handles_labels()[1]:
        ax.legend(fontsize=7, loc="best")
    style_ax(ax)

fig.suptitle("LC unit T-stats: RT encoding vs task-variable encoding", fontsize=14, y=0.98)
fig.tight_layout(rect=[0, 0, 1, 0.95])
save_fig(fig, "rt_vs_task_6panel", fig_dir=FIG_DIR, save=SAVE_FIG)
plt.show()

## 8. Build merged table (sue_plus) for T-scatter

In [ ]:
# T-scatter: RT vs outcome encoding, using the merge directly.
# Builds sue_plus for cell-level scatter + polar histogram.
def _merge_rt_and_sue(rt_stats, sue_df, get_session_prefix_fn):
    """Inner join RT T-stats onto sue_df on (session_prefix, unit)."""
    def canon(x):
        try: return str(int(float(x)))
        except Exception: return str(x)

    rt = rt_stats[["session_prefix","unit","T","coef","sig_fdr"]].copy()
    rt["unit"] = rt["unit"].map(canon)
    rt = rt.rename(columns={"T":"T_rt", "coef":"coef_rt", "sig_fdr":"sig_rt"})

    sue = sue_df.copy()
    if "unit_id" in sue.columns and "unit" not in sue.columns:
        sue = sue.rename(columns={"unit_id":"unit"})
    sue["unit"] = sue["unit"].map(canon)
    if "session_prefix" not in sue.columns:
        sue["session_prefix"] = sue["session"].astype(str).map(get_session_prefix_fn)
    sue["session_prefix"] = sue["session_prefix"].astype(str)

    return sue.merge(rt, on=["session_prefix","unit"], how="inner")

sue_plus = _merge_rt_and_sue(rt_result.stats, sue_features, get_session_prefix)
print(f"sue_plus shape: {sue_plus.shape}")
print(f"T_rt non-null: {sue_plus['T_rt'].notna().sum()}")

## 9. T-scatter + polar histogram

Each scatter: x = RT T-stat, y = Sue T-stat for one behavioral variable.
Polar histogram: distribution of coefficient vector angles, showing whether units
co-modulated by RT and the behavioral variable tend toward a preferred direction.

In [ ]:
# T-scatter + polar histogram for key comparison pairs.
def plot_T_scatter_and_polar(df, t_x_col, t_y_col, coef_x_col, coef_y_col,
                              *, polar_bins=12, figsize=(11, 5), title=None):
    """Scatter of T-stats + polar histogram of coefficient vector angles."""
    from scipy.stats import spearmanr

    mask = (df[t_x_col].notna() & df[t_y_col].notna() &
            df[coef_x_col].notna() & df[coef_y_col].notna())
    d = df[mask].copy()
    if len(d) < 5:
        print(f"Skipping {title}: too few points ({len(d)})")
        return None

    rho, p = spearmanr(d[t_x_col], d[t_y_col])
    angles = np.arctan2(d[coef_y_col].to_numpy(dtype=float),
                        d[coef_x_col].to_numpy(dtype=float))

    fig, (ax_s, ax_p) = plt.subplots(1, 2, figsize=figsize)

    # scatter
    ax_s.scatter(d[t_x_col], d[t_y_col], s=8, alpha=0.4, color=PALETTE["neutral"])
    ax_s.axhline(0, color="black", lw=0.6, ls="--")
    ax_s.axvline(0, color="black", lw=0.6, ls="--")
    ax_s.set_xlabel(t_x_col)
    ax_s.set_ylabel(t_y_col)
    ax_s.set_title(f"T-scatter  rho={rho:.3f}  p={p:.2g}  n={len(d)}")
    style_ax(ax_s)

    # polar
    ax_p = plt.subplot(1, 2, 2, projection="polar")
    edges = np.linspace(-np.pi, np.pi, polar_bins + 1)
    counts, _ = np.histogram(angles, bins=edges)
    width = edges[1] - edges[0]
    ax_p.bar(edges[:-1] + width / 2, counts, width=width * 0.9,
             color=PALETTE["accent"], alpha=0.7)
    ax_p.set_title("Coef angle (rad)", va="bottom")

    fig.suptitle(title or f"{t_x_col} vs {t_y_col}", fontsize=11)
    plt.tight_layout()
    return fig

# Key pairs: RT vs outcome, RT vs Q-chosen
PAIRS = [
    ("T_rt", "T_outcome_com_ori", "coef_rt", "coef_outcome_com_ori"),
    ("T_rt", "T_Qchosen_com_ori", "coef_rt", "coef_Qchosen_com_ori"),
    ("T_rt", "T_baseline_hit_all","coef_rt", "coef_baseline_hit_all"),
]
for tx, ty, cx, cy in PAIRS:
    if tx in sue_plus.columns and ty in sue_plus.columns:
        fig = plot_T_scatter_and_polar(
            sue_plus, tx, ty, cx, cy,
            title=f"{tx} vs {ty}",
        )
        if fig:
            save_fig(fig, f"t_scatter_{tx}_vs_{ty}", fig_dir=FIG_DIR, save=SAVE_FIG)
            plt.show()
    else:
        print(f"Skipping {tx}/{ty}: columns not found")